# Kazakh Fact-Checking Benchmark — прогон в Google Colab

Онлайн-прогон без терминала. Colab имеет открытый интернет, поэтому внешние API доступны.

## Порядок действий
1. Слева нажми значок 🔑 (**Secrets**) и добавь ключи (с тумблером **Notebook access**):
   - `GEMINI_API_KEY` — ключ Google AI Studio
   - `LLAMA_API_KEY` — ключ Groq (по желанию)
2. В ячейке «Настройки» выбери `SOURCE` (какой текст гоняем).
3. Выполняй ячейки сверху вниз (Shift+Enter).

⚠️ Ключи храни ТОЛЬКО в Secrets. Никогда не вставляй их в ячейки и не коммить в репозиторий.

### 1. Клонируем репозиторий и ставим зависимости

In [ ]:
BRANCH = 'claude/kazakh-factcheck-benchmark-e76kd5'
REPO = 'github.com/Tim2190/kazakh-factcheck-benchmark.git'
!rm -rf kazakh-factcheck-benchmark
!git clone --branch $BRANCH https://$REPO
%cd kazakh-factcheck-benchmark
!pip install -q -r requirements.txt
print('OK')

### 2. Настройки: какой текст гоняем
`leg_text01` — Конституция (большой, ~50k токенов). `news_text1` — новость (маленький, ~5k токенов).

In [ ]:
SOURCE = 'news_text1'   # поменяй на 'leg_text01' для текста 1

import os
from google.colab import userdata
for name in ['GEMINI_API_KEY', 'LLAMA_API_KEY']:
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val; print(f'{name}: загружен ({val[:4]}...)')
        else:
            print(f'{name}: пусто')
    except Exception:
        print(f'{name}: не найден в Secrets')

### 3. Прогон Gemini (батч-режим) + проверка заземления + скоринг
`--batch` шлёт весь документ + все тезисы одним запросом. Модель по умолчанию `gemini-2.5-flash`; если `429 / limit: 0` — поменяй `model_id` в `scripts/models.json` (напр. `gemini-3.1-flash-lite`).

In [ ]:
!python scripts/export_dataset.py
!python scripts/run_factcheck.py --model gemini --batch --source $SOURCE
!python scripts/check_grounding.py results/gemini_{SOURCE}_run.json
!python scripts/score.py results/gemini_{SOURCE}_run.json

### 4. (Опц.) Llama через Groq
Для маленького текста (`news_text1`) влезает в бесплатный Groq. Для большого (`leg_text01`) — упадёт `413`, гони Llama через веб-чат файлом `prompts/chat_run_<SOURCE>.txt`.

In [ ]:
!python scripts/run_factcheck.py --model llama --batch --source $SOURCE
!python scripts/check_grounding.py results/llama_{SOURCE}_run.json
!python scripts/score.py results/llama_{SOURCE}_run.json

### 5. Скачать результаты и прислать мне

In [ ]:
from google.colab import files
import glob
for f in glob.glob(f'results/*_{SOURCE}_run.json'):
    print('скачиваю', f); files.download(f)